# Turbofan RUL: tuning all nine models

Goal: lift every model's **R-Square above 0.7** and bring **RMSE and MAE down**, and keep a log of everything tried.

**How the tuning is kept honest**
- Same 70:30 engine split as `02_model_rf_lstm.ipynb` (seed 42). The 30 validation engines are **not used to pick anything**. Every choice below is made by 3-fold cross-validation over whole engines *inside the 70 training engines* (the LSTM uses one inner hold-out of 14 training engines instead, because it is slow to train).
- Each model is tuned in the same order, one change at a time: **features, then target form, then hyperparameters**. A change is kept only if it improves the cross-validated R-Square.
- Every trial is logged (model, change, settings, cross-validated R-Square / RMSE / MAE, time). The full log is shown in section 9 and saved to `artifacts/model/tuning_trials.csv`.
- The official test set stays closed. All scores are against the true, **uncapped** remaining life, as before.
- The final tables refit each model's chosen settings on all 70 training engines, then score the 70 training and the 30 validation engines.

**Disclosure.** Before writing this notebook I checked one XGBoost run on the validation engines to see whether extra features helped at all. The feature options below are all re-scored by cross-validation on training engines, and the final choice is made there, not on the validation engines.

In [2]:
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import BayesianRidge, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
import torch
from torch import nn
from xgboost import XGBRegressor

SEED = 42
VAL_FRACTION = 0.30
CV_FOLDS = 3
CAP_FOR_COMPARISON = 125     # only used for the secondary "capped truth" column
SAVE_OUTPUTS = True
N_JOBS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E4E7EB", "grid.linewidth": 0.6, "axes.axisbelow": True})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 400)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")


def find_project_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "csv_preview" / "train_FD001.csv").exists():
            return p
    raise FileNotFoundError("Could not find data/csv_preview/train_FD001.csv. Open the notebook from inside the project folder.")


ROOT = find_project_root()
print("Project root:", ROOT)

Project root: /Users/joseph/Desktop/RUL_prediction/TurbofanHPC_RUL_prediction


## 1. Data and the same 70:30 split

In [3]:
train_all = pd.read_csv(ROOT / "data" / "csv_preview" / "train_FD001.csv").sort_values(["engine_id", "cycle"]).reset_index(drop=True)
train_all["rul_cycles"] = train_all.groupby("engine_id")["cycle"].transform("max") - train_all["cycle"]
eda = json.loads((ROOT / "artifacts" / "eda" / "eda_split_and_columns.json").read_text())
SENSORS = [c for c in eda["keep"] if c.startswith("sensor")]


def split_engines(engine_ids, val_fraction, seed):
    ids = np.array(sorted({int(e) for e in engine_ids}))
    rng = np.random.default_rng(seed)
    n_val = max(1, int(round(len(ids) * val_fraction)))
    val = np.sort(rng.choice(ids, size=n_val, replace=False))
    return [int(x) for x in np.setdiff1d(ids, val)], [int(x) for x in val]


dev_ids, val_ids = split_engines(train_all["engine_id"].unique(), VAL_FRACTION, SEED)
assert set(dev_ids).isdisjoint(val_ids)
print(f"{len(dev_ids)} training engines, {len(val_ids)} validation engines. Sensors kept: {len(SENSORS)}")

70 training engines, 30 validation engines. Sensors kept: 12


## 2. Features: the original set and three richer sets

All features are **causal**: a row only uses that engine's current and earlier cycles, so they work the same in live use. The leakage check below proves it.

| Feature set | What is in it |
|---|---|
| `base` | The 60 features used before: 12 sensors, plus rolling mean and std over 5 and 10 cycles |
| `base+cycle` | `base` plus the cycle number (how long the engine has run) |
| `extended` | `base+cycle` plus: each sensor minus that engine's own early level (mean of its first 5 cycles), an exponentially weighted mean (span 20), and a 20-cycle rolling mean |
| `ext+poly2` | `extended` plus pairwise products of the cycle number and the 12 ten-cycle sensor means. Tried for the linear models only |

The "own early level" feature matters because every engine starts at a slightly different level, so a sensor's *change from its own start* is a cleaner wear signal than its raw value.

In [4]:
def build_features(df):
    df = df.sort_values(["engine_id", "cycle"]).reset_index(drop=True)
    g = df.groupby("engine_id")[SENSORS]
    cols = {}
    for w in (5, 10):
        roll = g.rolling(window=w, min_periods=1)
        mean = roll.mean().reset_index(level=0, drop=True).sort_index()
        std = roll.std(ddof=0).reset_index(level=0, drop=True).sort_index()
        for s in SENSORS:
            cols[f"{s}_mean_{w}"] = mean[s]
            cols[f"{s}_std_{w}"] = std[s]
    # engine's own early level: running mean for the first 5 cycles, then the mean of cycles 1-5 (all in the past)
    early = g.transform(lambda x: x.expanding().mean().where(np.arange(len(x)) < 5).ffill())
    ewm = g.transform(lambda x: x.ewm(span=20, adjust=False).mean())
    roll20 = g.rolling(window=20, min_periods=1).mean().reset_index(level=0, drop=True).sort_index()
    for s in SENSORS:
        cols[f"{s}_vs_start"] = df[s] - early[s]
        cols[f"{s}_ewm20"] = ewm[s]
        cols[f"{s}_mean_20"] = roll20[s]
    out = pd.concat([df, pd.DataFrame(cols)], axis=1)
    base_cols = ["cycle"] + [f"{s}_mean_10" for s in SENSORS]
    poly = {}
    for i, a in enumerate(base_cols):
        for b in base_cols[i:]:
            poly[f"poly_{a}__{b}"] = out[a] * out[b]
    return pd.concat([out, pd.DataFrame(poly)], axis=1)


feats = build_features(train_all)

# leakage check: chopping off the future must not change earlier rows' features
one = train_all[train_all["engine_id"] == dev_ids[0]]
full = build_features(one)
cut = build_features(one.iloc[:60])
shared = [c for c in full.columns if c not in ("rul_cycles",)]
assert np.allclose(full.iloc[:60][shared].to_numpy(float), cut[shared].to_numpy(float)), "features use future rows"
print("Leakage check passed: all features at cycle 60 are identical with or without later cycles.")

FS = {}
FS["base"] = list(SENSORS) + [c for c in feats.columns if c.startswith("sensor_") and ("_mean_5" in c or "_mean_10" in c or "_std_" in c)]
FS["base+cycle"] = FS["base"] + ["cycle"]
FS["extended"] = FS["base+cycle"] + [c for c in feats.columns if c.endswith("_vs_start") or c.endswith("_ewm20") or c.endswith("_mean_20")]
FS["ext+poly2"] = FS["extended"] + [c for c in feats.columns if c.startswith("poly_")]
for k, v in FS.items():
    assert len(v) == len(set(v)) and all(c in feats.columns for c in v)
print({k: len(v) for k, v in FS.items()})

dev_df = feats[feats["engine_id"].isin(dev_ids)].reset_index(drop=True)
val_df = feats[feats["engine_id"].isin(val_ids)].reset_index(drop=True)
print(f"training rows {len(dev_df):,} | validation rows {len(val_df):,}")

Leakage check passed: all features at cycle 60 are identical with or without later cycles.
{'base': 60, 'base+cycle': 61, 'extended': 97, 'ext+poly2': 188}
training rows 14,128 | validation rows 6,503


## 3. The tuning machinery

- `TARGETS` are the target forms tried: the raw remaining life, square-root and log forms (which let linear models bend toward the end of life), and a cap at 150 or 125 cycles (a common trick that stops models chasing early-life noise). Predictions are always turned back into cycles and scored against the **uncapped** truth, so a cap has to earn its place.
- `tune()` follows the fixed order: baseline, then features, then target, then hyperparameters, keeping only changes that raise cross-validated R-Square.

In [5]:
def score(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)
    e = p - y
    return {"R2": r2_score(y, p), "RMSE": float(np.sqrt(np.mean(e ** 2))), "MAE": float(np.mean(np.abs(e)))}


TARGETS = {
    "raw": (lambda y: y, lambda z: z),
    "sqrt": (np.sqrt, np.square),
    "log1p": (np.log1p, np.expm1),
    "cap150": (lambda y: np.minimum(y, 150), lambda z: z),
    "cap125": (lambda y: np.minimum(y, 125), lambda z: z),
}

folds = list(GroupKFold(n_splits=CV_FOLDS).split(dev_df, groups=dev_df["engine_id"]))
y_dev = dev_df["rul_cycles"].to_numpy(float)
y_val = val_df["rul_cycles"].to_numpy(float)
TRIALS = []


def fit_predict(make, params, fs, target, train_rows, test_rows):
    cols = FS[fs]
    fwd, inv = TARGETS[target]
    model = make(params, len(cols))
    model.fit(train_rows[cols].to_numpy(float), fwd(train_rows["rul_cycles"].to_numpy(float)))
    return np.clip(inv(model.predict(test_rows[cols].to_numpy(float))), 0, None)


def run_trial(name, stage, change, make, params, fs, target, best_r2=None):
    t0 = time.time()
    oof = np.zeros(len(dev_df))
    for tr, te in folds:
        oof[te] = fit_predict(make, params, fs, target, dev_df.iloc[tr], dev_df.iloc[te])
    m = score(y_dev, oof)
    new_best = best_r2 is None or m["R2"] > best_r2 + 1e-4
    rec = {"model": name, "stage": stage, "change": change, "features": fs, "n_features": len(FS[fs]), "target": target,
           "settings": json.dumps(params, default=str), "cv_R2": m["R2"], "cv_RMSE": m["RMSE"], "cv_MAE": m["MAE"],
           "seconds": round(time.time() - t0, 1), "outcome": "baseline" if best_r2 is None else ("kept (new best)" if new_best else "not better")}
    TRIALS.append(rec)
    print(f"  [{stage:<15}] {change:<46} R2 {m['R2']:.3f}  RMSE {m['RMSE']:6.2f}  MAE {m['MAE']:6.2f}  {rec['seconds']:5.1f}s  -> {rec['outcome']}")
    return rec, new_best


def tune(name, make, base_params, fs_options, target_options, param_candidates, base_fs="base"):
    print(f"\n=== {name} ===")
    rec, _ = run_trial(name, "0 baseline", "previous notebook settings", make, base_params, base_fs, "raw")
    best = {"fs": base_fs, "target": "raw", "params": dict(base_params), "r2": rec["cv_R2"]}
    for fs in fs_options:
        rec, nb_ = run_trial(name, "1 features", f"feature set: {fs}", make, best["params"], fs, best["target"], best["r2"])
        if nb_:
            best.update(fs=fs, r2=rec["cv_R2"])
    for tg in target_options:
        rec, nb_ = run_trial(name, "2 target", f"target form: {tg}", make, best["params"], best["fs"], tg, best["r2"])
        if nb_:
            best.update(target=tg, r2=rec["cv_R2"])
    for cand in param_candidates:
        params = {**best["params"], **cand}
        rec, nb_ = run_trial(name, "3 hyperparameters", ", ".join(f"{k}={v}" for k, v in cand.items()), make, params, best["fs"], best["target"], best["r2"])
        if nb_:
            best.update(params=params, r2=rec["cv_R2"])
    print(f"  BEST for {name}: features={best['fs']}, target={best['target']}, params={best['params']}, CV R2={best['r2']:.3f}")
    return best, dict(fs=base_fs, target="raw", params=dict(base_params))


BEST, BEFORE = {}, {}

## 4. Linear Regression, Bayesian Ridge and Logistic Regression

Linear models can only draw a straight-line relationship, so their levers are the inputs (features, products of features) and the target form. For plain Linear Regression the only hyperparameter is an optional ridge penalty (`alpha`), tried last. For Logistic Regression the levers are the band width and the regularisation strength `C`.

In [6]:
MAKERS = {}
MAKERS["Linear Regression"] = lambda p, nf: make_pipeline(StandardScaler(), Ridge(alpha=p["alpha"]) if p["alpha"] > 0 else LinearRegression())
BEST["Linear Regression"], BEFORE["Linear Regression"] = tune(
    "Linear Regression", MAKERS["Linear Regression"], {"alpha": 0},
    fs_options=["base+cycle", "extended", "ext+poly2"], target_options=["sqrt", "log1p", "cap150", "cap125"],
    param_candidates=[{"alpha": 1}, {"alpha": 10}, {"alpha": 100}, {"alpha": 1000}])


=== Linear Regression ===
  [0 baseline     ] previous notebook settings                     R2 0.588  RMSE  43.67  MAE  32.20    0.1s  -> baseline
  [1 features     ] feature set: base+cycle                        R2 0.637  RMSE  40.97  MAE  30.31    0.1s  -> kept (new best)


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

  [1 features     ] feature set: extended                          R2 0.577  RMSE  44.21  MAE  31.41    0.1s  -> not better


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

  [1 features     ] feature set: ext+poly2                         R2 0.614  RMSE  42.27  MAE  29.43    0.3s  -> not better
  [2 target       ] target form: sqrt                              R2 0.644  RMSE  40.54  MAE  28.54    0.1s  -> kept (new best)
  [2 target       ] target form: log1p                             R2 0.427  RMSE  51.45  MAE  35.69    0.1s  -> not better
  [2 target       ] target form: cap150                            R2 0.581  RMSE  43.99  MAE  30.74    0.1s  -> not better


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

  [2 target       ] target form: cap125                            R2 0.484  RMSE  48.86  MAE  33.52    0.1s  -> not better
  [3 hyperparameters] alpha=1                                        R2 0.644  RMSE  40.54  MAE  28.54    0.0s  -> not better
  [3 hyperparameters] alpha=10                                       R2 0.645  RMSE  40.54  MAE  28.53    0.0s  -> kept (new best)
  [3 hyperparameters] alpha=100                                      R2 0.643  RMSE  40.60  MAE  28.59    0.0s  -> not better
  [3 hyperparameters] alpha=1000                                     R2 0.632  RMSE  41.24  MAE  29.15    0.0s  -> not better
  BEST for Linear Regression: features=base+cycle, target=sqrt, params={'alpha': 10}, CV R2=0.645


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarn

In [7]:
MAKERS["Bayesian Ridge"] = lambda p, nf: make_pipeline(StandardScaler(), BayesianRidge(**p))
BEST["Bayesian Ridge"], BEFORE["Bayesian Ridge"] = tune(
    "Bayesian Ridge", MAKERS["Bayesian Ridge"], {},
    fs_options=["base+cycle", "extended", "ext+poly2"], target_options=["sqrt", "log1p", "cap150", "cap125"],
    param_candidates=[{"alpha_1": 1e-3, "lambda_1": 1e-3}, {"alpha_1": 1.0, "lambda_1": 1.0},
                      {"alpha_2": 1e-3, "lambda_2": 1e-3}, {"max_iter": 600, "tol": 1e-5}])


=== Bayesian Ridge ===
  [0 baseline     ] previous notebook settings                     R2 0.587  RMSE  43.68  MAE  32.20    0.1s  -> baseline
  [1 features     ] feature set: base+cycle                        R2 0.637  RMSE  40.94  MAE  30.30    0.1s  -> kept (new best)


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

  [1 features     ] feature set: extended                          R2 0.584  RMSE  43.87  MAE  31.16    0.1s  -> not better


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

  [1 features     ] feature set: ext+poly2                         R2 0.628  RMSE  41.45  MAE  28.50    0.4s  -> not better
  [2 target       ] target form: sqrt                              R2 0.645  RMSE  40.54  MAE  28.53    0.1s  -> kept (new best)
  [2 target       ] target form: log1p                             R2 0.428  RMSE  51.43  MAE  35.69    0.1s  -> not better


...

/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

  [3 hyperparameters] alpha_1=0.001, lambda_1=0.001                  R2 0.645  RMSE  40.54  MAE  28.53    0.1s  -> not better
  [3 hyperparameters] alpha_1=1.0, lambda_1=1.0                      R2 0.645  RMSE  40.54  MAE  28.53    0.1s  -> not better


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

  [3 hyperparameters] alpha_2=0.001, lambda_2=0.001                  R2 0.645  RMSE  40.54  MAE  28.53    0.1s  -> not better
  [3 hyperparameters] max_iter=600, tol=1e-05                        R2 0.645  RMSE  40.54  MAE  28.53    0.1s  -> not better
  BEST for Bayesian Ridge: features=base+cycle, target=sqrt, params={}, CV R2=0.645


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

In [8]:
class LogitBands:
    # Logistic regression on bands of remaining life; the prediction is the probability-weighted band centre
    def __init__(self, band=10, C=1.0):
        self.band, self.C = band, C

    def fit(self, X, y):
        self.scaler = StandardScaler().fit(X)
        bands = (np.asarray(y) // self.band).astype(int)
        self.clf = LogisticRegression(C=self.C, max_iter=300, tol=1e-3).fit(self.scaler.transform(X), bands)
        self.centres = self.clf.classes_ * self.band + (self.band - 1) / 2.0
        return self

    def predict(self, X):
        return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres


MAKERS["Logistic Regression (bands)"] = lambda p, nf: LogitBands(**p)
BEST["Logistic Regression (bands)"], BEFORE["Logistic Regression (bands)"] = tune(
    "Logistic Regression (bands)", MAKERS["Logistic Regression (bands)"], {"band": 10, "C": 1.0},
    fs_options=["base+cycle", "extended"], target_options=["cap150"],
    param_candidates=[{"band": 20}, {"C": 0.1}, {"C": 10.0}])


=== Logistic Regression (bands) ===


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/joseph/Desktop/RUL_prediction/.

  [0 baseline     ] previous notebook settings                     R2 0.601  RMSE  42.94  MAE  30.58    1.0s  -> baseline


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: divide by zero encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: overflow encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeW

  [1 features     ] feature set: base+cycle                        R2 0.661  RMSE  39.62  MAE  27.61    1.0s  -> kept (new best)


...

/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: divide by zero encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: overflow encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeW

  [3 hyperparameters] C=0.1                                          R2 0.723  RMSE  35.79  MAE  23.59    1.2s  -> kept (new best)


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: divide by zero encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: overflow encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeW

  [3 hyperparameters] C=10.0                                         R2 0.680  RMSE  38.48  MAE  24.47    2.2s  -> not better
  BEST for Logistic Regression (bands): features=extended, target=raw, params={'band': 10, 'C': 0.1}, CV R2=0.723


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: divide by zero encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: overflow encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeW

## 5. SVM (SVR) and Decision Tree

SVR levers: `C` (how hard it fits), `epsilon_frac` (the no-penalty tube, as a fraction of the target spread, so it stays sensible under square-root or log targets) and `gamma_mult` (how local the RBF kernel is). The tree's levers are depth and the minimum rows per leaf.

In [9]:
class SVRRel:
    # SVR whose epsilon is set relative to the spread of the (possibly transformed) target
    def __init__(self, nf, C=100.0, epsilon_frac=1.0 / 68.0, gamma_mult=1.0):
        self.nf, self.C, self.epsilon_frac, self.gamma_mult = nf, C, epsilon_frac, gamma_mult

    def fit(self, X, y):
        self.scaler = StandardScaler().fit(X)
        eps = self.epsilon_frac * float(np.std(y))
        self.m = SVR(kernel="rbf", C=self.C, epsilon=eps, gamma=self.gamma_mult / self.nf).fit(self.scaler.transform(X), y)
        return self

    def predict(self, X):
        return self.m.predict(self.scaler.transform(X))


MAKERS["SVM (SVR)"] = lambda p, nf: SVRRel(nf, **p)
BEST["SVM (SVR)"], BEFORE["SVM (SVR)"] = tune(
    "SVM (SVR)", MAKERS["SVM (SVR)"], {"C": 100.0, "epsilon_frac": 1.0 / 68.0, "gamma_mult": 1.0},
    fs_options=["base+cycle", "extended"], target_options=["sqrt", "cap150"],
    param_candidates=[{"C": 30.0}, {"C": 300.0}, {"gamma_mult": 0.3}, {"gamma_mult": 3.0}, {"epsilon_frac": 0.05}])


=== SVM (SVR) ===
  [0 baseline     ] previous notebook settings                     R2 0.580  RMSE  44.08  MAE  30.52   19.1s  -> baseline
  [1 features     ] feature set: base+cycle                        R2 0.646  RMSE  40.47  MAE  27.14   20.2s  -> kept (new best)
  [1 features     ] feature set: extended                          R2 0.678  RMSE  38.58  MAE  24.50   21.7s  -> kept (new best)
  [2 target       ] target form: sqrt                              R2 0.638  RMSE  40.89  MAE  26.52  126.1s  -> not better
  [2 target       ] target form: cap150                            R2 0.650  RMSE  40.25  MAE  25.23   25.8s  -> not better
  [3 hyperparameters] C=30.0                                         R2 0.698  RMSE  37.37  MAE  23.20   15.6s  -> kept (new best)
  [3 hyperparameters] C=300.0                                        R2 0.644  RMSE  40.59  MAE  26.57   51.0s  -> not better
  [3 hyperparameters] gamma_mult=0.3                                 R2 0.705  RMSE  36.93  MAE 

In [10]:
MAKERS["Decision Tree"] = lambda p, nf: DecisionTreeRegressor(random_state=SEED, **p)
BEST["Decision Tree"], BEFORE["Decision Tree"] = tune(
    "Decision Tree", MAKERS["Decision Tree"], {"max_depth": 10, "min_samples_leaf": 10},
    fs_options=["base+cycle", "extended"], target_options=["sqrt", "cap150"],
    param_candidates=[{"max_depth": 6}, {"max_depth": 8}, {"max_depth": 12}, {"max_depth": None},
                      {"min_samples_leaf": 5}, {"min_samples_leaf": 20}, {"min_samples_leaf": 40}])


=== Decision Tree ===
  [0 baseline     ] previous notebook settings                     R2 0.445  RMSE  50.65  MAE  35.38    0.9s  -> baseline
  [1 features     ] feature set: base+cycle                        R2 0.517  RMSE  47.24  MAE  32.28    0.9s  -> kept (new best)
  [1 features     ] feature set: extended                          R2 0.577  RMSE  44.23  MAE  29.41    1.5s  -> kept (new best)
  [2 target       ] target form: sqrt                              R2 0.597  RMSE  43.18  MAE  28.30    1.5s  -> kept (new best)
  [2 target       ] target form: cap150                            R2 0.610  RMSE  42.44  MAE  27.91    1.5s  -> kept (new best)
  [3 hyperparameters] max_depth=6                                    R2 0.617  RMSE  42.07  MAE  27.86    1.0s  -> kept (new best)
  [3 hyperparameters] max_depth=8                                    R2 0.612  RMSE  42.34  MAE  27.90    1.3s  -> not better
  [3 hyperparameters] max_depth=12                                   R2 0.608  RMS

## 6. Random Forest and XGBoost

Random Forest levers: rows per leaf, and how many features each split may consider (`max_features`). XGBoost levers: tree depth, learning rate with more trees, minimum child weight, sampling and regularisation. During tuning the forest uses 200 trees to save time; the final model uses 300, as before.

In [11]:
MAKERS["Random Forest"] = lambda p, nf: RandomForestRegressor(n_estimators=p.get("n_estimators", 200), random_state=SEED, n_jobs=N_JOBS,
                                                              **{k: v for k, v in p.items() if k != "n_estimators"})
BEST["Random Forest"], BEFORE["Random Forest"] = tune(
    "Random Forest", MAKERS["Random Forest"], {"min_samples_leaf": 5, "max_features": 1.0},
    fs_options=["base+cycle", "extended"], target_options=["sqrt", "cap150"],
    param_candidates=[{"max_features": 0.5}, {"max_features": 0.3}, {"min_samples_leaf": 2}, {"min_samples_leaf": 10},
                      {"min_samples_leaf": 20}, {"max_depth": 15}])


=== Random Forest ===
  [0 baseline     ] previous notebook settings                     R2 0.596  RMSE  43.24  MAE  30.60   81.0s  -> baseline
  [1 features     ] feature set: base+cycle                        R2 0.666  RMSE  39.32  MAE  26.98   76.6s  -> kept (new best)
  [1 features     ] feature set: extended                          R2 0.708  RMSE  36.76  MAE  24.01  126.9s  -> kept (new best)
  [2 target       ] target form: sqrt                              R2 0.718  RMSE  36.12  MAE  23.51  125.6s  -> kept (new best)
  [2 target       ] target form: cap150                            R2 0.657  RMSE  39.85  MAE  24.91  121.4s  -> not better
  [3 hyperparameters] max_features=0.5                               R2 0.725  RMSE  35.63  MAE  23.14   62.3s  -> kept (new best)
  [3 hyperparameters] max_features=0.3                               R2 0.726  RMSE  35.57  MAE  23.20   40.0s  -> kept (new best)
  [3 hyperparameters] min_samples_leaf=2                             R2 0.725  RMS

In [12]:
MAKERS["XGBoost"] = lambda p, nf: XGBRegressor(random_state=SEED, n_jobs=N_JOBS, **p)
BEST["XGBoost"], BEFORE["XGBoost"] = tune(
    "XGBoost", MAKERS["XGBoost"],
    {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 5, "subsample": 0.8, "colsample_bytree": 0.8},
    fs_options=["base+cycle", "extended"], target_options=["sqrt", "cap150"],
    param_candidates=[{"max_depth": 3}, {"max_depth": 4}, {"max_depth": 7}, {"min_child_weight": 10}, {"min_child_weight": 30},
                      {"colsample_bytree": 0.5}, {"reg_lambda": 10.0}, {"n_estimators": 800, "learning_rate": 0.025}])


=== XGBoost ===
  [0 baseline     ] previous notebook settings                     R2 0.596  RMSE  43.24  MAE  30.37    2.9s  -> baseline
  [1 features     ] feature set: base+cycle                        R2 0.668  RMSE  39.15  MAE  27.13    2.8s  -> kept (new best)
  [1 features     ] feature set: extended                          R2 0.719  RMSE  36.03  MAE  23.68    4.3s  -> kept (new best)
  [2 target       ] target form: sqrt                              R2 0.725  RMSE  35.64  MAE  22.91    4.4s  -> kept (new best)
  [2 target       ] target form: cap150                            R2 0.671  RMSE  39.00  MAE  24.05    4.5s  -> not better
  [3 hyperparameters] max_depth=3                                    R2 0.716  RMSE  36.24  MAE  23.60    1.9s  -> not better
  [3 hyperparameters] max_depth=4                                    R2 0.716  RMSE  36.21  MAE  23.26    2.8s  -> not better
  [3 hyperparameters] max_depth=7                                    R2 0.729  RMSE  35.40  MAE  2

## 7. LSTM

The LSTM is slow to train, so it is tuned on **one inner hold-out of 14 of the 70 training engines** instead of 3-fold cross-validation. That score is noisier than the others, which is a limit on how fine the tuning can be. Each trial changes one thing from the best setting so far: add the cycle number as an input, a longer look-back window, a bigger network, longer training with a decaying learning rate, and more dropout with weight decay.

In [13]:
LSTM_BASE = dict(win=30, hidden=64, layers=2, dropout=0.1, epochs=30, lr=1e-3, wd=0.0, sched=False, use_cycle=False, batch=256)
rng_inner = np.random.default_rng(SEED)
inner_val_ids = sorted(int(x) for x in rng_inner.choice(dev_ids, size=14, replace=False))
inner_train_ids = [e for e in dev_ids if e not in inner_val_ids]


class LSTMNet(nn.Module):
    def __init__(self, n_in, hidden, layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_in, hidden, num_layers=layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def make_seqs(df, scaler, win, use_cycle):
    parts = []
    for _, g in df.groupby("engine_id", sort=True):
        g = g.sort_values("cycle")
        x = scaler.transform(g[SENSORS].to_numpy(float))
        if use_cycle:
            x = np.hstack([x, (g["cycle"].to_numpy(float) / 300.0)[:, None]])
        padded = np.vstack([np.repeat(x[:1], win - 1, axis=0), x])
        parts.append(np.lib.stride_tricks.sliding_window_view(padded, (win, x.shape[1]))[:, 0].astype(np.float32))
    out = np.concatenate(parts)
    assert len(out) == len(df)
    return out


def train_lstm(train_rows, cfg, seed=SEED):
    scaler = StandardScaler().fit(train_rows[SENSORS].to_numpy(float))
    X = torch.from_numpy(make_seqs(train_rows, scaler, cfg["win"], cfg["use_cycle"]))
    y = torch.from_numpy((train_rows["rul_cycles"].to_numpy(float) / 100.0).astype(np.float32))
    torch.manual_seed(seed)
    net = LSTMNet(X.shape[2], cfg["hidden"], cfg["layers"], cfg["dropout"])
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"]) if cfg["sched"] else None
    gen = torch.Generator().manual_seed(seed)
    for _ in range(cfg["epochs"]):
        net.train()
        order = torch.randperm(len(X), generator=gen)
        for i in range(0, len(order), cfg["batch"]):
            idx = order[i:i + cfg["batch"]]
            opt.zero_grad()
            loss = nn.functional.mse_loss(net(X[idx]), y[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        if sched:
            sched.step()
    return net, scaler


@torch.no_grad()
def lstm_predict(model, rows, cfg):
    net, scaler = model
    net.eval()
    seqs = make_seqs(rows, scaler, cfg["win"], cfg["use_cycle"])
    out = [net(torch.from_numpy(seqs[i:i + 1024])).numpy() for i in range(0, len(seqs), 1024)]
    return np.clip(np.concatenate(out) * 100.0, 0, None)


inner_tr = train_all[train_all["engine_id"].isin(inner_train_ids)]
inner_te = train_all[train_all["engine_id"].isin(inner_val_ids)]
inner_te = inner_te.sort_values(["engine_id", "cycle"])
y_inner = inner_te["rul_cycles"].to_numpy(float)
print(f"LSTM inner split: {len(inner_train_ids)} engines to train, {len(inner_val_ids)} engines to score")


def lstm_trial(stage, change, cfg, best_r2=None):
    t0 = time.time()
    model = train_lstm(inner_tr, cfg)
    m = score(y_inner, lstm_predict(model, inner_te, cfg))
    new_best = best_r2 is None or m["R2"] > best_r2 + 1e-4
    rec = {"model": "LSTM", "stage": stage, "change": change, "features": "12 sensors" + (" + cycle" if cfg["use_cycle"] else ""),
           "n_features": len(SENSORS) + int(cfg["use_cycle"]), "target": "raw", "settings": json.dumps(cfg),
           "cv_R2": m["R2"], "cv_RMSE": m["RMSE"], "cv_MAE": m["MAE"], "seconds": round(time.time() - t0, 1),
           "outcome": "baseline" if best_r2 is None else ("kept (new best)" if new_best else "not better")}
    TRIALS.append(rec)
    print(f"  [{stage:<15}] {change:<46} R2 {m['R2']:.3f}  RMSE {m['RMSE']:6.2f}  MAE {m['MAE']:6.2f}  {rec['seconds']:5.1f}s  -> {rec['outcome']}")
    return rec, new_best


print("\n=== LSTM (score = inner hold-out of 14 training engines, not 3-fold CV) ===")
rec, _ = lstm_trial("0 baseline", "previous notebook settings", LSTM_BASE)
lstm_best = {"cfg": dict(LSTM_BASE), "r2": rec["cv_R2"]}

LSTM inner split: 56 engines to train, 14 engines to score

=== LSTM (score = inner hold-out of 14 training engines, not 3-fold CV) ===
  [0 baseline     ] previous notebook settings                     R2 0.609  RMSE  35.02  MAE  24.42   35.8s  -> baseline


In [14]:
lstm_candidates = [
    ("1 features", "add cycle number as an input", {"use_cycle": True}),
    ("3 hyperparameters", "look-back window 50", {"win": 50}),
    ("3 hyperparameters", "hidden size 128", {"hidden": 128}),
    ("3 hyperparameters", "60 epochs, cosine learning-rate decay", {"epochs": 60, "sched": True}),
    ("3 hyperparameters", "dropout 0.2 and weight decay 1e-4", {"dropout": 0.2, "wd": 1e-4}),
]
for stage, change, patch in lstm_candidates:
    cfg = {**lstm_best["cfg"], **patch}
    rec, nb_ = lstm_trial(stage, change, cfg, lstm_best["r2"])
    if nb_:
        lstm_best = {"cfg": cfg, "r2": rec["cv_R2"]}
print("  BEST LSTM settings:", lstm_best["cfg"], f"| inner hold-out R2 {lstm_best['r2']:.3f}")
BEST["LSTM"], BEFORE["LSTM"] = lstm_best, {"cfg": dict(LSTM_BASE)}

  [1 features     ] add cycle number as an input                   R2 0.683  RMSE  31.54  MAE  21.33   34.8s  -> kept (new best)
  [3 hyperparameters] look-back window 50                            R2 0.629  RMSE  34.13  MAE  22.61   57.1s  -> not better
  [3 hyperparameters] hidden size 128                                R2 0.638  RMSE  33.71  MAE  22.54   65.8s  -> not better
  [3 hyperparameters] 60 epochs, cosine learning-rate decay          R2 0.636  RMSE  33.80  MAE  22.12   67.7s  -> not better
  [3 hyperparameters] dropout 0.2 and weight decay 1e-4              R2 0.646  RMSE  33.31  MAE  22.48   37.0s  -> not better
  BEST LSTM settings: {'win': 30, 'hidden': 64, 'layers': 2, 'dropout': 0.1, 'epochs': 30, 'lr': 0.001, 'wd': 0.0, 'sched': False, 'use_cycle': True, 'batch': 256} | inner hold-out R2 0.683


## 8. Refit the chosen settings and score every model

Each model is refit on all 70 training engines (the "before" settings and the "after" tuned settings), then scored on the 70 training engines and on the 30 validation engines. This is the first and only time the validation engines are used.

In [15]:
MODEL_ORDER = ["Mean baseline", "Linear Regression", "Logistic Regression (bands)", "Bayesian Ridge",
               "SVM (SVR)", "Decision Tree", "Random Forest", "XGBoost", "LSTM"]
FINAL_TREES = {"n_estimators": 300}
preds = {"before": {"train": {}, "val": {}}, "after": {"train": {}, "val": {}}}
mean_rul = y_dev.mean()
for stage in preds:
    preds[stage]["train"]["Mean baseline"] = np.full(len(dev_df), mean_rul)
    preds[stage]["val"]["Mean baseline"] = np.full(len(val_df), mean_rul)

t0 = time.time()
for name in MODEL_ORDER[1:]:
    for stage, cfg in (("before", BEFORE[name]), ("after", BEST[name])):
        if name == "LSTM":
            c = cfg["cfg"]
            model = train_lstm(dev_df, c)
            preds[stage]["train"][name] = lstm_predict(model, dev_df, c)
            preds[stage]["val"][name] = lstm_predict(model, val_df, c)
        else:
            params = dict(cfg["params"])
            if name == "Random Forest":
                params.update(FINAL_TREES)
            for part, rows in (("train", dev_df), ("val", val_df)):
                pass
            cols = FS[cfg["fs"]]
            fwd, inv = TARGETS[cfg["target"]]
            m = MAKERS[name](params, len(cols))
            m.fit(dev_df[cols].to_numpy(float), fwd(y_dev))
            preds[stage]["train"][name] = np.clip(inv(m.predict(dev_df[cols].to_numpy(float))), 0, None)
            preds[stage]["val"][name] = np.clip(inv(m.predict(val_df[cols].to_numpy(float))), 0, None)
    print(f"{name:<28} refit done ({time.time() - t0:.0f} s elapsed)")

for stage in preds:
    for part, n in (("train", len(dev_df)), ("val", len(val_df))):
        for name in MODEL_ORDER:
            p = preds[stage][part][name]
            assert len(p) == n and np.isfinite(p).all(), f"bad predictions: {stage} {part} {name}"
print("All predictions are finite and the right length.")

/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/joseph/Desktop/RUL_predict

Linear Regression            refit done (0 s elapsed)


/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: divide by zero encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeWarning: overflow encountered in matmul
  return self.clf.predict_proba(self.scaler.transform(X)) @ self.centres
/var/folders/gn/rxwhz96n7f7dqnmjm8p0nwgr0000gn/T/ipykernel_80449/3484310570.py:14: RuntimeW

Logistic Regression (bands)  refit done (1 s elapsed)
Bayesian Ridge               refit done (1 s elapsed)
SVM (SVR)                    refit done (54 s elapsed)
Decision Tree                refit done (55 s elapsed)
Random Forest                refit done (141 s elapsed)
XGBoost                      refit done (143 s elapsed)
LSTM                         refit done (205 s elapsed)
All predictions are finite and the right length.


In [16]:
def table(stage, part):
    y = y_dev if part == "train" else y_val
    rows = {n: score(y, preds[stage][part][n]) for n in MODEL_ORDER}
    return pd.DataFrame(rows).T.rename(columns={"R2": "R-Square"}).rename_axis("model")


tbl = {(s, p): table(s, p) for s in ("before", "after") for p in ("train", "val")}

# before vs after on the validation engines, with the pass/fail check on R-Square
cmp_ = pd.DataFrame({
    "R-Square before": tbl[("before", "val")]["R-Square"], "R-Square after": tbl[("after", "val")]["R-Square"],
    "RMSE before": tbl[("before", "val")]["RMSE"], "RMSE after": tbl[("after", "val")]["RMSE"],
    "MAE before": tbl[("before", "val")]["MAE"], "MAE after": tbl[("after", "val")]["MAE"],
})
cmp_["R-Square above 0.7?"] = np.where(cmp_["R-Square after"] > 0.7, "yes", "NO")
cmp_.loc["Mean baseline", "R-Square above 0.7?"] = "-"
cmp_["RMSE and MAE lower?"] = np.where((cmp_["RMSE after"] < cmp_["RMSE before"]) & (cmp_["MAE after"] < cmp_["MAE before"]), "yes", "NO")
cmp_.loc["Mean baseline", "RMSE and MAE lower?"] = "-"
print("VALIDATION ENGINES, before vs after tuning (true uncapped remaining life):\n")
display(cmp_)

# secondary view: the same predictions against remaining life capped at 125 cycles (a convention used in many papers)
cap_view = pd.DataFrame({n: {"R-Square vs capped-125 truth": r2_score(np.minimum(y_val, CAP_FOR_COMPARISON),
                                                                      np.minimum(preds["after"]["val"][n], CAP_FOR_COMPARISON))}
                         for n in MODEL_ORDER}).T
print("\nSecondary view, tuned models: validation R-Square when both the truth and the prediction are capped at 125 cycles")
print("(many published results use this easier scoring; the table above is the stricter one):")
display(cap_view)

VALIDATION ENGINES, before vs after tuning (true uncapped remaining life):



,R-Square before,R-Square after,RMSE before,RMSE after,MAE before,MAE after,R-Square above 0.7?,RMSE and MAE lower?
model,,,,,,,,
Mean baseline,-0.009,-0.009,70.859,70.859,58.809,58.809,-,-
Linear Regression,0.623,0.682,43.278,39.764,32.105,28.784,NO,yes
Logistic Regression (bands),0.617,0.757,43.676,34.782,31.473,23.918,yes,yes
Bayesian Ridge,0.623,0.682,43.320,39.765,32.129,28.786,NO,yes
SVM (SVR),0.568,0.713,46.367,37.765,33.603,26.035,yes,yes
Decision Tree,0.507,0.600,49.545,44.590,35.904,30.990,NO,yes
Random Forest,0.621,0.731,43.445,36.552,31.527,25.889,yes,yes
XGBoost,0.618,0.734,43.591,36.400,31.663,25.318,yes,yes
LSTM,0.634,0.680,42.689,39.876,29.142,27.951,NO,yes



Secondary view, tuned models: validation R-Square when both the truth and the prediction are capped at 125 cycles
(many published results use this easier scoring; the table above is the stricter one):


,R-Square vs capped-125 truth
Mean baseline,-0.168
Linear Regression,0.817
Logistic Regression (bands),0.902
Bayesian Ridge,0.817
SVM (SVR),0.884
Decision Tree,0.829
Random Forest,0.894
XGBoost,0.905
LSTM,0.851


### Save the tuned Logistic Regression, Random Forest and XGBoost models

Refits the three tuned models on the 70 training engines, saves each to `model_save/` (a compressed `.joblib` bundle holding the model, its feature list, target form and settings), then reloads each file and rescores the validation engines to prove the saved copy works. `src/rul_models.py` holds the small helper needed to load the logistic model and turn predictions back into cycles.

In [17]:
import sys
import platform
import joblib
import sklearn
import xgboost

sys.path.insert(0, str(ROOT))                      # makes the project's src/ package importable
from src.rul_models import LogitBands, predict_rul

SAVE_DIR = ROOT / "model_save"
SAVE_DIR.mkdir(exist_ok=True)

SAVE_SPEC = {
    "Logistic Regression (bands)": ("logistic_regression_bands", lambda p: LogitBands(**p)),
    "Random Forest": ("random_forest", lambda p: RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=N_JOBS, **p)),
    "XGBoost": ("xgboost", lambda p: XGBRegressor(random_state=SEED, n_jobs=N_JOBS, **p)),
}
saved = []
for name, (stem, make_model) in SAVE_SPEC.items():
    b = BEST[name]
    cols = FS[b["fs"]]
    fwd, _ = TARGETS[b["target"]]
    model = make_model(b["params"]).fit(dev_df[cols].to_numpy(float), fwd(y_dev))
    bundle = {
        "name": name, "model": model, "features": cols, "feature_set": b["fs"], "target": b["target"],
        "settings": b["params"], "train_engine_ids": dev_ids, "seed": SEED,
        "versions": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                     "xgboost": xgboost.__version__, "joblib": joblib.__version__},
    }
    path = SAVE_DIR / f"{stem}.joblib"
    joblib.dump(bundle, path, compress=3)         # compressed so the Random Forest file stays small
    # reload from disk and score the validation engines, to prove the saved file works on its own
    again = joblib.load(path)
    m = score(y_val, predict_rul(again, val_df))
    saved.append({"model": name, "file": f"model_save/{stem}.joblib", "size (MB)": path.stat().st_size / 1e6,
                  "features": f"{b['fs']} ({len(cols)})", "target": b["target"],
                  "reloaded R-Square": m["R2"], "RMSE": m["RMSE"], "MAE": m["MAE"]})
display(pd.DataFrame(saved).set_index("model"))
print("The reloaded models should match the validation table in section 11.")

(SAVE_DIR / "README.md").write_text(
    "# model_save\n\n"
    "Tuned models from `model/03_model_tuning.ipynb`, each trained on the 70 training engines (seed 42, 70:30 split).\n\n"
    "| File | Model |\n|---|---|\n"
    "| `logistic_regression_bands.joblib` | Logistic Regression on 10-cycle bands |\n"
    "| `random_forest.joblib` | Random Forest, 300 trees |\n"
    "| `xgboost.joblib` | XGBoost, 800 trees, square-root target |\n\n"
    "Each file is a dictionary with the fitted `model`, its `features` (column names, in order), the `target` form, the tuned `settings`, "
    "`train_engine_ids` and the library `versions` it was saved with.\n\n"
    "```python\n"
    "import sys, joblib\n"
    "sys.path.insert(0, '<project root>')            # so 'src' can be imported\n"
    "from src.rul_models import predict_rul         # also needed to unpickle the logistic model\n"
    "bundle = joblib.load('model_save/xgboost.joblib')\n"
    "rul = predict_rul(bundle, feature_frame)       # feature_frame must hold the columns in bundle['features']\n"
    "```\n\n"
    "The feature columns are built by `build_features()` in section 2 of the notebook (causal rolling features). "
    "Load with the same scikit-learn and XGBoost versions listed in each bundle, or expect version warnings.\n"
)
print("Saved to", SAVE_DIR)

/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/joseph/Desktop/RUL_prediction/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * weights
/Users/joseph/Desktop/RUL_prediction/.

,file,size (MB),features,target,reloaded R-Square,RMSE,MAE
model,,,,,,,
Logistic Regression (bands),model_save/logistic_regression_bands.joblib,0.032,extended (97),raw,0.757,34.782,23.918
Random Forest,model_save/random_forest.joblib,22.105,extended (97),sqrt,0.731,36.552,25.889
XGBoost,model_save/xgboost.joblib,0.679,extended (97),sqrt,0.734,36.400,25.318


The reloaded models should match the validation table in section 11.
Saved to /Users/joseph/Desktop/RUL_prediction/TurbofanHPC_RUL_prediction/model_save


## 9. Everything that was tried

One row per trial. `cv_R2`, `cv_RMSE` and `cv_MAE` come from 3-fold cross-validation over whole training engines (LSTM: the 14-engine inner hold-out). `outcome` says whether the change was kept. Sorted in the order the trials ran.

In [18]:
trials = pd.DataFrame(TRIALS)
trials.insert(0, "trial", range(1, len(trials) + 1))
if SAVE_OUTPUTS:
    out_dir = ROOT / "artifacts" / "model"
    out_dir.mkdir(parents=True, exist_ok=True)
    trials.to_csv(out_dir / "tuning_trials.csv", index=False)
    cmp_.to_csv(out_dir / "tuned_vs_original_validation.csv")
    json.dump({n: {k: v for k, v in BEST[n].items() if k != "r2"} for n in BEST},
              open(out_dir / "tuned_settings.json", "w"), indent=2, default=str)

print(f"{len(trials)} trials in total ({trials['seconds'].sum() / 60:.0f} min of fitting). Saved to artifacts/model/tuning_trials.csv\n")
show = ["trial", "stage", "change", "features", "target", "cv_R2", "cv_RMSE", "cv_MAE", "outcome"]
for name in MODEL_ORDER[1:]:
    sub = trials[trials["model"] == name]
    print(f"--- {name}: {len(sub)} trials ---")
    display(sub[show].reset_index(drop=True))

83 trials in total (25 min of fitting). Saved to artifacts/model/tuning_trials.csv

--- Linear Regression: 12 trials ---


,trial,stage,change,features,target,cv_R2,cv_RMSE,cv_MAE,outcome
0,1,0 baseline,previous notebook settings,base,raw,0.588,43.667,32.196,baseline
1,2,1 features,feature set: base+cycle,base+cycle,raw,0.637,40.974,30.307,kept (new best)
2,3,1 features,feature set: extended,extended,raw,0.577,44.210,31.413,not better
3,4,1 features,feature set: ext+poly2,ext+poly2,raw,0.614,42.274,29.431,not better
4,5,2 target,target form: sqrt,base+cycle,sqrt,0.644,40.544,28.544,kept (new best)
5,6,2 target,target form: log1p,base+cycle,log1p,0.427,51.452,35.692,not better
6,7,2 target,target form: cap150,base+cycle,cap150,0.581,43.994,30.741,not better
7,8,2 target,target form: cap125,base+cycle,cap125,0.484,48.856,33.520,not better
8,9,3 hyperparameters,alpha=1,base+cycle,sqrt,0.644,40.543,28.542,not better
9,10,3 hyperparameters,alpha=10,base+cycle,sqrt,0.645,40.537,28.535,kept (new best)


--- Logistic Regression (bands): 7 trials ---


,trial,stage,change,features,target,cv_R2,cv_RMSE,cv_MAE,outcome
0,25,0 baseline,previous notebook settings,base,raw,0.601,42.940,30.579,baseline
1,26,1 features,feature set: base+cycle,base+cycle,raw,0.661,39.619,27.611,kept (new best)
2,27,1 features,feature set: extended,extended,raw,0.708,36.741,23.490,kept (new best)
3,28,2 target,target form: cap150,extended,cap150,0.660,39.623,24.518,not better
4,29,3 hyperparameters,band=20,extended,raw,0.702,37.099,23.919,not better
5,30,3 hyperparameters,C=0.1,extended,raw,0.723,35.791,23.586,kept (new best)
6,31,3 hyperparameters,C=10.0,extended,raw,0.680,38.483,24.467,not better


--- Bayesian Ridge: 12 trials ---


...

,trial,stage,change,features,target,cv_R2,cv_RMSE,cv_MAE,outcome
0,54,0 baseline,previous notebook settings,base,raw,0.596,43.240,30.602,baseline
1,55,1 features,feature set: base+cycle,base+cycle,raw,0.666,39.320,26.982,kept (new best)
2,56,1 features,feature set: extended,extended,raw,0.708,36.764,24.006,kept (new best)
3,57,2 target,target form: sqrt,extended,sqrt,0.718,36.116,23.507,kept (new best)
4,58,2 target,target form: cap150,extended,cap150,0.657,39.849,24.914,not better
5,59,3 hyperparameters,max_features=0.5,extended,sqrt,0.725,35.634,23.142,kept (new best)
6,60,3 hyperparameters,max_features=0.3,extended,sqrt,0.726,35.571,23.204,kept (new best)
7,61,3 hyperparameters,min_samples_leaf=2,extended,sqrt,0.725,35.660,23.186,not better
8,62,3 hyperparameters,min_samples_leaf=10,extended,sqrt,0.724,35.742,23.426,not better
9,63,3 hyperparameters,min_samples_leaf=20,extended,sqrt,0.724,35.707,23.548,not better


--- XGBoost: 13 trials ---


,trial,stage,change,features,target,cv_R2,cv_RMSE,cv_MAE,outcome
0,65,0 baseline,previous notebook settings,base,raw,0.596,43.239,30.368,baseline
1,66,1 features,feature set: base+cycle,base+cycle,raw,0.668,39.152,27.127,kept (new best)
2,67,1 features,feature set: extended,extended,raw,0.719,36.029,23.676,kept (new best)
3,68,2 target,target form: sqrt,extended,sqrt,0.725,35.635,22.914,kept (new best)
4,69,2 target,target form: cap150,extended,cap150,0.671,38.999,24.050,not better
5,70,3 hyperparameters,max_depth=3,extended,sqrt,0.716,36.242,23.596,not better
6,71,3 hyperparameters,max_depth=4,extended,sqrt,0.716,36.215,23.264,not better
7,72,3 hyperparameters,max_depth=7,extended,sqrt,0.729,35.399,22.666,kept (new best)
8,73,3 hyperparameters,min_child_weight=10,extended,sqrt,0.729,35.402,22.690,not better
9,74,3 hyperparameters,min_child_weight=30,extended,sqrt,0.731,35.265,22.702,kept (new best)


--- LSTM: 6 trials ---


,trial,stage,change,features,target,cv_R2,cv_RMSE,cv_MAE,outcome
0,78,0 baseline,previous notebook settings,12 sensors,raw,0.609,35.017,24.421,baseline
1,79,1 features,add cycle number as an input,12 sensors + cycle,raw,0.683,31.539,21.330,kept (new best)
2,80,3 hyperparameters,look-back window 50,12 sensors + cycle,raw,0.629,34.133,22.614,not better
3,81,3 hyperparameters,hidden size 128,12 sensors + cycle,raw,0.638,33.713,22.541,not better
4,82,3 hyperparameters,"60 epochs, cosine learning-rate decay",12 sensors + cycle,raw,0.636,33.798,22.120,not better
5,83,3 hyperparameters,dropout 0.2 and weight decay 1e-4,12 sensors + cycle,raw,0.646,33.311,22.483,not better


### What each model ended up with

In [19]:
rows = []
for name in MODEL_ORDER[1:]:
    b = BEST[name]
    sub = trials[trials["model"] == name]
    base_r2 = sub.iloc[0]["cv_R2"]
    if name == "LSTM":
        chosen = {k: v for k, v in b["cfg"].items() if LSTM_BASE[k] != v}
        rows.append({"model": name, "features": "12 sensors" + (" + cycle" if b["cfg"]["use_cycle"] else ""), "target": "raw",
                     "changed settings": json.dumps(chosen), "tuning score before": base_r2, "tuning score after": b["r2"]})
    else:
        base_params = BEFORE[name]["params"]
        chosen = {k: v for k, v in b["params"].items() if base_params.get(k) != v}
        rows.append({"model": name, "features": b["fs"], "target": b["target"], "changed settings": json.dumps(chosen, default=str),
                     "tuning score before": base_r2, "tuning score after": b["r2"]})
display(pd.DataFrame(rows).set_index("model"))
print("'tuning score' = cross-validated R-Square on training engines (LSTM: inner hold-out of 14 training engines).")

,features,target,changed settings,tuning score before,tuning score after
model,,,,,
Linear Regression,base+cycle,sqrt,"{""alpha"": 10}",0.588,0.645
Logistic Regression (bands),extended,raw,"{""C"": 0.1}",0.601,0.723
Bayesian Ridge,base+cycle,sqrt,{},0.587,0.645
SVM (SVR),extended,raw,"{""C"": 30.0, ""epsilon_frac"": 0.05, ""gamma_mult"": 0.3}",0.580,0.706
Decision Tree,extended,cap150,"{""max_depth"": 6}",0.445,0.617
Random Forest,extended,sqrt,"{""max_features"": 0.3}",0.596,0.726
XGBoost,extended,sqrt,"{""max_depth"": 7, ""colsample_bytree"": 0.5, ""min_child_weight"": 30, ""reg_lambda"": 10.0}",0.596,0.733
LSTM,12 sensors + cycle,raw,"{""use_cycle"": true}",0.609,0.683


'tuning score' = cross-validated R-Square on training engines (LSTM: inner hold-out of 14 training engines).


## 10. Final results: training set (tuned models)

R-Square (higher is better), RMSE and MAE in cycles (lower is better), on the 70 training engines. These models learned from these engines, so this is the optimistic view.

In [20]:
print(f"TRAINING SET: {len(dev_ids)} engines, {len(dev_df):,} rows. Tuned models.\n")
display(tbl[("after", "train")])

TRAINING SET: 70 engines, 14,128 rows. Tuned models.



,R-Square,RMSE,MAE
model,,,
Mean baseline,0.000,67.999,55.837
Linear Regression,0.668,39.197,27.455
Logistic Regression (bands),0.791,31.082,20.595
Bayesian Ridge,0.668,39.199,27.456
SVM (SVR),0.782,31.737,18.528
Decision Tree,0.659,39.725,24.897
Random Forest,0.972,11.431,6.904
XGBoost,0.986,7.936,5.366
LSTM,0.953,14.665,10.636


## 11. Final results: validation set (tuned models)

The same three measures on the 30 validation engines that no model trained on or was tuned on. This is the table to compare models on.

In [21]:
val_after = tbl[("after", "val")]
print(f"VALIDATION SET: {len(val_ids)} engines, {len(val_df):,} rows. Tuned models.\n")
display(val_after)
ranked = val_after.drop(index="Mean baseline")
below = [n for n in ranked.index if ranked.loc[n, "R-Square"] <= 0.7]
print(f"\nBest R-Square: {ranked['R-Square'].idxmax()} | lowest RMSE: {ranked['RMSE'].idxmin()} | lowest MAE: {ranked['MAE'].idxmin()}")
print("Models above R-Square 0.7:", ", ".join(n for n in ranked.index if n not in below) or "none")
print("Models NOT above 0.7:", ", ".join(below) or "none")

VALIDATION SET: 30 engines, 6,503 rows. Tuned models.



,R-Square,RMSE,MAE
model,,,
Mean baseline,-0.009,70.859,58.809
Linear Regression,0.682,39.764,28.784
Logistic Regression (bands),0.757,34.782,23.918
Bayesian Ridge,0.682,39.765,28.786
SVM (SVR),0.713,37.765,26.035
Decision Tree,0.600,44.590,30.990
Random Forest,0.731,36.552,25.889
XGBoost,0.734,36.400,25.318
LSTM,0.680,39.876,27.951



Best R-Square: Logistic Regression (bands) | lowest RMSE: Logistic Regression (bands) | lowest MAE: Logistic Regression (bands)
Models above R-Square 0.7: Logistic Regression (bands), SVM (SVR), Random Forest, XGBoost
Models NOT above 0.7: Linear Regression, Bayesian Ridge, Decision Tree, LSTM
